# 02 — Merge LoRA + 4-bit GPTQ quantization

Run **00_setup_colab.ipynb** and **01_sft_train_qlora.ipynb** first in this same session (or reload a previously saved/pushed adapter below).

GPTQ quantizes dense weights, not LoRA adapters, so the adapter has to be merged into the base model first. Uses **GPTQModel** (AutoGPTQ is deprecated as of 2026).

## Load config

In [ ]:
from slm_prod.utils import load_config

model_cfg = load_config("model.yaml")
sft_cfg = load_config("sft.yaml")
gptq_cfg = load_config("gptq.yaml")


## (If starting a fresh session) reload the adapter

Skip this cell if `model`/`tokenizer` from notebook 01 are still in memory.

In [ ]:
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=sft_cfg["adapter_dir"],  # or your pushed hub repo id
#     max_seq_length=model_cfg["max_seq_length"],
#     load_in_4bit=True,
# )


## Merge adapter into full-precision base weights

In [ ]:
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_cfg["base_model_id"], torch_dtype=torch.bfloat16, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(sft_cfg["adapter_dir"])

merged = PeftModel.from_pretrained(base_model, sft_cfg["adapter_dir"])
merged = merged.merge_and_unload()

merged_dir = Path(sft_cfg["merged_dir"])
merged_dir.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(merged_dir))
tokenizer.save_pretrained(str(merged_dir))
print(f"Merged model saved to {merged_dir}")

del base_model, merged
torch.cuda.empty_cache()


## Quantize to 4-bit GPTQ

In [ ]:
from slm_prod.quantize_gptq import load_calibration_texts
from gptqmodel import GPTQModel, QuantizeConfig

quantize_config = QuantizeConfig(
    bits=gptq_cfg["bits"],
    group_size=gptq_cfg["group_size"],
    desc_act=gptq_cfg["desc_act"],
    sym=gptq_cfg["sym"],
    damp_percent=gptq_cfg["damp_percent"],
)

gptq_model = GPTQModel.load(gptq_cfg["input_dir"], quantize_config)
calibration_texts = load_calibration_texts(gptq_cfg)
gptq_model.quantize(calibration_texts, batch_size=1)

gptq_model.save(gptq_cfg["output_dir"])
print(f"GPTQ 4-bit model saved to {gptq_cfg['output_dir']}")


## Sanity-check generation from the quantized model

In [ ]:
from gptqmodel import GPTQModel

reloaded = GPTQModel.load(gptq_cfg["output_dir"])
prompt = "Explain what a GPTQ quantized model is, in two sentences, to a non-technical reader."
print(reloaded.generate(prompt, max_new_tokens=120))


## Optional: push both checkpoints to the Hub

```python
# !python scripts/push_to_hub.py {merged_dir} your-username/gemma-4-e4b-no_robots-sft
# !python scripts/push_to_hub.py {gptq_cfg['output_dir']} your-username/gemma-4-e4b-no_robots-sft-gptq4bit
```

Continue with **03_eval_base_sft_gptq.ipynb**.